In [14]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [15]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [16]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [17]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(index=index, llm_client=openai_client, instructions=instructions)
assistant.rag("How do I run Ollama locally")
assistant.rag("How do I run Olama locally?")

'There is no information in the provided CONTEXT about how to run Olama locally. The CONTEXT does mention running the course locally in Module 1: RAG, but it does not provide specific instructions for running Olama.'

In [18]:
messages = [{"role": "user", "content": "How do I run Ollama locally?"}]

response = openai_client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=messages,
)
print(response.choices[0].message.content)


<think>
Here's a thinking process:

1.  **Understand User Query**: The user wants to know how to run Ollama locally. This is a straightforward technical question about installing and using Ollama, a popular open-source tool for running large language models locally.

2.  **Identify Key Information Needed**:
   - What is Ollama? (Brief context, optional but helpful)
   - System requirements/compatibility
   - Installation steps for major OS (macOS, Windows, Linux)
   - Basic usage (running a model, pulling models, API/server mode)
   - Troubleshooting/common tips

3.  **Verify Current Information (as of 2024/2025)**:
   - Ollama supports macOS, Windows, and Linux
   - Installation is typically via a single command or a straightforward installer
   - Core commands: `ollama pull <model>`, `ollama run <model>`, `ollama serve`
   - Default server runs on localhost:11434
   - No GPU required to run, but highly recommended for performance
   - Official docs: https://ollama.com

4.  **Structu

In [19]:
messages = [{"role": "user", "content": "How do I run Ollama locally?"}]

response = openai_client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=messages,
    tools=[search_tool],
)
print(response.choices[0].message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='fm23cr063', function=Function(arguments='{"query":"run Ollama locally"}', name='search'), type='function')]


In [20]:
import json

tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
results = search(**args)
result_json = json.dumps(results, indent=2)

messages.append({
    "role": "assistant",
    "tool_calls": [{
        "id": tool_call.id,
        "type": "function",
        "function": {
            "name": tool_call.function.name,
            "arguments": tool_call.function.arguments
        }
    }]
})

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result_json,
})

response = openai_client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=messages,
    tools=[search_tool],
)

print(response.choices[0].message.content)

To run Ollama locally, follow these installation and usage steps based on your operating system:

### 1. Install Ollama
Visit [https://ollama.com/download](https://ollama.com/download) and download the version for your OS:
*   **macOS:** Download the `.pkg` file and install it.
*   **Windows:** Download the `.msi` file and install it.
*   **Linux:** Run the following command in your terminal:
    ```bash
    curl -fsSL https://ollama.com/install.sh | sh
    ```

### 2. Run a Model
Once installed, open your terminal and run a model (e.g., Llama 3) using this command:
```bash
ollama run llama3
```
This command will download the model (approx. 4GB), start it locally, and open a chat interface.

### 3. Verify the Server
To test that the local server is running, run:
```bash
curl http://localhost:11434
```
You should see a response similar to `{"models": [...]}`.

### 4. Use with Python (Optional)
If you want to interact with Ollama via code, first install the Python client:
```bash
pip ins